In [1]:
# ========== 导入：微调「改进前沿模型」练习所需依赖 ==========
# 练习目标：把商品 Item 编成 JSONL → 上传 → 开 fine-tune job → 跟踪 loss → 在验证集上测 MAE

# os / pickle / json / time / random：环境、序列化数据、等待作业、打乱样本
import os
import pickle
import json
import time
import random
# OpenAI 官方客户端：Files + Fine-tuning + Chat Completions
from openai import OpenAI
# 画训练曲线、算误差
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
# 课程本地模块：Item 数据结构；Tester 评测（本笔记本后面主要手写评估）
from items import Item 
from testing import Tester  


In [2]:
# ========== 创建 OpenAI 客户端 ==========

# 从环境变量 OPENAI_API_KEY 读密钥（需事先 export 或在 shell/.env 配好）
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [3]:
# ========== 微调超参：集中写在常量里，后面 jobs.create 直接引用 ==========

# 基座模型 ID：字符串必须与 OpenAI 支持的 fine-tune 型号一致
BASE_MODEL = "gpt-4o-mini-2024-07-18"
# 训练轮数
EPOCHS = 5
# 批大小
BATCH_SIZE = 8
# 学习率乘数（learning_rate_multiplier）
LR_MULT = 0.3


In [ ]:
# ========== 加载本地 pickle：训练子集 + 验证/测试子集 ==========

# train.pkl：训练用 Item 列表（二进制协议，rb 打开）
with open("train.pkl", "rb") as f:
    train_subset = pickle.load(f)

# 文件名是 test.pkl，变量却叫 val_subset：沿用作者命名，后面评估也用它
with open("test.pkl", "rb") as f:
    val_subset = pickle.load(f) 

print(f"Loaded {len(train_subset)} training and {len(val_subset)} validation items.")


In [ ]:
# ========== 打乱并截断：控制微调成本与耗时 ==========

# 原地打乱，避免原始顺序带来偏差
random.shuffle(train_subset)
random.shuffle(val_subset)

# 只用前 N 条；想全量可改大（费用与时间上升）
TRAIN_LIMIT = 5000 
VAL_LIMIT = 1000    

train_subset = train_subset[:TRAIN_LIMIT]
val_subset = val_subset[:VAL_LIMIT]

print(f"Using {len(train_subset)} training and {len(val_subset)} validation samples.")


In [6]:
# ========== Prompt / Completion 模板：决定模型「怎么学定价」==========
# 注意：f-string 里的英文指令与示例是可运行 prompt，保持原文不翻译

# build_prompt：拼出 user 侧长提示（语境 / 任务 / few-shot / 当前商品字段）
def build_prompt(item):
    return f"""
#＃＃ 语境
You are a price estimation assistant for e-commerce listings.
Each product is described by its title, category, key features, and details.

#＃＃ 任务
Estimate the most likely retail price in USD.
Think step-by-step about product type, quality, and included components 
before stating the final answer as "Predicted Price: $<amount>".

### 示例
- Wireless earbuds with active noise cancellation -> Predicted Price: $89
- Stainless steel kitchen knife set (6-piece) -> Predicted Price: $45
- Laptop stand aluminum adjustable -> Predicted Price: $32

### 产品标题
{item.title}

#＃＃ 类别
{item.category}

#＃＃ 细节
{item.details}

### 你的推理
(Think about product quality, features, and typical market range.)

### 最终答案
Predicted Price: $
"""

# build_completion：assistant 标签，格式对齐 "Predicted Price: $..."
def build_completion(item):
    return f"Predicted Price: ${round(item.price)}.00"


In [ ]:
# ========== 写 JSONL：OpenAI chat 微调格式 ==========

def write_jsonl(data, filename):
    # utf-8 文本文件；每行一个 JSON 对象
    with open(filename, "w", encoding="utf-8") as f:
        for item in data:
            # 若 Item 带 include 标志则尊重；没有该属性则默认纳入
            if getattr(item, "include", True):
                prompt = build_prompt(item)
                completion = build_completion(item)
                # messages：user=提示，assistant=标准答案
                json_obj = {
                    "messages": [
                        {"role": "user", "content": prompt},
                        {"role": "assistant", "content": completion}
                    ]
                }
                f.write(json.dumps(json_obj) + "\n")
    print(f"Wrote {len(data)} samples to {filename}")

# 训练 / 验证 JSONL 文件名（相对当前工作目录）
TRAIN_JSONL = "train_prepared.jsonl"
VAL_JSONL = "val_prepared.jsonl"

# 各写一份，供下一格 files.create 上传
write_jsonl(train_subset, TRAIN_JSONL)
write_jsonl(val_subset, VAL_JSONL)


In [8]:
# ========== 上传微调文件到 OpenAI Files API ==========

# purpose="fine-tune"：声明这些文件将用于微调（不是普通存储）
train_file = client.files.create(file=open(TRAIN_JSONL, "rb"), purpose="fine-tune")
val_file = client.files.create(file=open(VAL_JSONL, "rb"), purpose="fine-tune")


In [ ]:
# ========== 创建 fine-tuning job ==========

job = client.fine_tuning.jobs.create(
    # 上一格返回的 file id
    training_file=train_file.id,
    validation_file=val_file.id,
    # 基座模型与超参常量
    model=BASE_MODEL,
    hyperparameters={
        "n_epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate_multiplier": LR_MULT
    }
)

# 记下 job.id，下一格轮询事件流要用
print("Job started:", job.id)


In [ ]:
# ========== 跟踪微调事件：打印日志 + 可选画 loss 曲线 ==========

def stream_finetune_events(job_id, poll_interval=30):
    print(f"Tracking fine-tuning job: {job_id}\n")
    # seen：已打印过的 event id，避免重复刷屏
    seen = set()
    # (step, train_loss, val_loss|None) 列表，供画图
    loss_data = []
    
    while True:
        # 拉最新 job 状态
        job = client.fine_tuning.jobs.retrieve(job_id)
        # 拉事件列表（新事件可能插在前面，下面倒序遍历）
        events = client.fine_tuning.jobs.list_events(job_id)
        
        for e in events.data[::-1]:
            if e.id not in seen:
                seen.add(e.id)
                # created_at 是 unix 时间戳 → 本地可读时间
                ts = datetime.fromtimestamp(e.created_at)
                msg = e.message
                print(f"[{ts:%Y-%m-%d %H:%M:%S}] {msg}")
                
                # 从事件文案里抠 training_loss / val_loss（格式依赖 OpenAI 日志字符串）
                if "training_loss" in msg:
                    try:
                        step = int(msg.split("Step ")[1].split("/")[0])
                        train_loss = float(msg.split("training_loss: ")[1].split(",")[0])
                        val_loss = None
                        if "val_loss" in msg:
                            val_loss = float(msg.split("val_loss: ")[1].split(",")[0])
                        loss_data.append((step, train_loss, val_loss))
                    except Exception:
                        # 解析失败就跳过该条，不中断轮询
                        pass
        
        if job.status == "succeeded":
            print("\nFine-tuning complete!")
            print("Fine-tuned model ID:", job.fine_tuned_model)
            
            if loss_data:
                steps = [d[0] for d in loss_data]
                train_losses = [d[1] for d in loss_data]
                # 只保留有 val_loss 的点
                val_losses = [d[2] for d in loss_data if d[2] is not None]

                plt.figure(figsize=(8, 5))
                plt.plot(steps, train_losses, marker="o", color="teal", label="Training Loss")
                if val_losses:
                    plt.plot(steps[:len(val_losses)], val_losses, marker="o", color="orange", label="Validation Loss")
                plt.xlabel("Step")
                plt.ylabel("Loss")
                plt.title(f"Fine-Tuning Progress — {job_id}")
                plt.legend()
                plt.grid(alpha=0.3)
                plt.show()
            else:
                print("No loss data found. Fine-tuning may have completed too quickly to log metrics.")

            # 成功：返回微调后模型 ID，供后续 chat 调用
            return job.fine_tuned_model

        elif job.status in ["failed", "cancelled"]:
            print(f"\nFine-tuning {job.status}.")
            if job.error:
                print("Error:", job.error)
            return None

        # 未结束：睡一会再问，默认 30 秒
        time.sleep(poll_interval)

# 阻塞直到结束；MODEL_ID 可能是 ft:... 或 None
MODEL_ID = stream_finetune_events(job.id)


In [ ]:
# ========== 用微调模型在验证子集上试定价 ==========

def test_model(model_id, test_items, max_samples=100):
    y_true, y_pred = [], []
    for i, item in enumerate(test_items[:max_samples]):
        # 与训练时同一套 build_prompt，保证「训练格式 = 推理格式」
        prompt = build_prompt(item)
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        output = response.choices[0].message.content
        try:
            # 约定输出含 $price：取第一个 $ 后的数字
            pred_price = float(output.split("$")[1].split()[0])
        except:
            # 解析失败则跳过该样本（不计入 y_true/y_pred）
            continue
        y_true.append(item.price)
        y_pred.append(pred_price)
        print(f"{i+1}. {item.title[:50]} | Actual: ${item.price} | Pred: ${pred_price}")
    return y_true, y_pred

# 对上一格得到的 MODEL_ID 跑验证子集
y_true, y_pred = test_model(MODEL_ID, val_subset)


In [ ]:
# ========== 可视化：按误差上色的预测 vs 真值散点 ==========

# 逐样本绝对误差
errors = np.abs(np.array(y_true) - np.array(y_pred))
# <10 绿，<25 橙，否则红 —— 一眼看出哪些样本偏得多
colors = ["green" if e < 10 else "orange" if e < 25 else "red" for e in errors]

plt.figure(figsize=(10,6))
# 蓝点=真实价；彩色点=预测价（颜色=误差档）
plt.scatter(range(len(y_true)), y_true, color='blue', label='Actual', alpha=0.6)
plt.scatter(range(len(y_pred)), y_pred, color=colors, label='Predicted', alpha=0.8)
plt.title("Fine-tuned Price Prediction Performance (Color-Coded by Error)")
plt.xlabel("Sample Index")
plt.ylabel("Price ($)")
plt.legend()
plt.show()

# 平均绝对误差摘要
avg_error = np.mean(errors)
print(f"\nAverage error: ${avg_error:.2f}")
